# Multi-ISI Forgetting Curve Analysis

Population-level analysis of auditory memory across ISI conditions.

**Analyses:**
1. d' vs ISI forgetting curves (with optional delay-condition overlay)
2. Hit rate and false-alarm rate by ISI
3. Split-half reliability of the d' curve
4. Random-split replication
5. Inter-response timing diagnostics
6. Stimulus frequency distributions

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt

from utls.loading import load_results_with_exclusion_no_dropping, move_sequences_to_used, split_by_musicianship
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability

from utls.data_loading import load_multi_isi, HR_TASK_NAMES, make_save_dir
from utls.human_analysis import (
    run_analysis, split_half_reliability, inter_response_times, p_to_stars,
)
from utls.human_plotting import (
    plot_dprime_vs_isi, plot_hit_and_fa_rates, plot_split_half_histogram,
    plot_inter_response_times, plot_random_split, plot_stimulus_frequency,
)

## Configuration

Change the parameters below to select the task and delay conditions.

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
TASK = "atexts"          # Options: "env-sounds", "glob-music", "atexts", "nhs-region-len120"
DELAYS = [None, 2, 4]    # None = no delay, 2 = 2s, 4 = 4s  (set to [None] for single condition)
BATCH_SIZE = 8

FIGURES_BASE = "/om2/user/bjmedina/auditory-memory/memory/figures/human-results"
TASK_HR_NAME = HR_TASK_NAMES[TASK]

print(f"Task: {TASK_HR_NAME}")
print(f"Delay conditions: {DELAYS}")

## 1. Load Data

In [ ]:
# Load each delay condition
datasets = {}
for delay in DELAYS:
    data = load_multi_isi(
        task=TASK,
        load_fn=load_results_with_exclusion_no_dropping,
        delay=delay,
        min_dprime=2,
        min_trials=120,
        batch_size=BATCH_SIZE,
    )
    label = f"{delay}s delay" if delay else "No delay"
    datasets[label] = data
    print(f"{label}: N={data['N']} participants")
    if data['batch_info']:
        print(f"  Complete batches: {data['batch_info']['complete_batches']}")

In [ ]:
# Compute per-subject d' and run analysis for each condition
analyses = {}
dprime_dfs = {}

for label, data in datasets.items():
    x = recompute_dprime_by_isi_per_subject(data['exps'])
    dprime_dfs[label] = x
    analyses[label] = run_analysis(x)
    print(f"{label}: {analyses[label]['N']} subjects, "
          f"d'(ISI=0) = {analyses[label]['dprime'][0]:.2f}, "
          f"d'(ISI=64) = {analyses[label]['dprime'][-1]:.2f}")

## 2. d' vs ISI (All Delay Conditions Overlaid)

In [ ]:
results_list = list(analyses.values())
labels = [f"{lbl} (N={a['N']})" for lbl, a in analyses.items()]
colors = ['g', 'b', 'orange', 'red', 'purple'][:len(results_list)]

save_dir = make_save_dir(FIGURES_BASE, TASK, sub="multi-isi")

plot_dprime_vs_isi(
    results_list,
    labels=labels,
    colors=colors,
    title=f"{TASK_HR_NAME}: Population d' vs ISI",
    save_path=os.path.join(save_dir, "dprime-vs-isi_all-delays.png"),
)

## 3. Hit Rate & False-Alarm Rate by ISI

In [ ]:
plot_hit_and_fa_rates(
    results_list,
    labels=labels,
    colors=colors,
    title=f"{TASK_HR_NAME}: Hit Rate & False Alarm Rate",
    save_path=os.path.join(save_dir, "hit-rate_and_fa-all-delays.png"),
)

## 4. Split-Half Reliability of d' Curve

Uses the no-delay (or first) condition.

In [ ]:
# Use the primary (first) condition for reliability
primary_label = list(dprime_dfs.keys())[0]
x_primary = dprime_dfs[primary_label]

rel = split_half_reliability(x_primary, n_splits=2000, method="pearson")
print(f"Mean split-half r = {rel['mean_r']:.3f}")
print(f"95% CI: [{rel['ci'][0]:.3f}, {rel['ci'][1]:.3f}]")

plot_split_half_histogram(
    rel,
    title=f"{TASK_HR_NAME}: Split-Half Reliability of d' vs ISI",
    save_path=os.path.join(save_dir, "dprime-consistency.png"),
)

## 5. Random Split Replication

In [ ]:
plot_random_split(
    x_primary,
    run_analysis,
    title=f"{TASK_HR_NAME}: d' vs ISI (Random Split)",
    save_path=os.path.join(save_dir, "dprime-vs-isi_random-split.png"),
)

## 6. Inter-Response Timing Diagnostics

In [ ]:
primary_exps = datasets[primary_label]['exps']
irts = inter_response_times(primary_exps)

print(f"Mean IRT: {np.mean(irts):.0f} ms")
print(f"Max IRT: {np.max(irts):.0f} ms")

plot_inter_response_times(
    irts,
    title=f"Inter-Response Time Distribution (mean={np.mean(irts):.0f} ms)",
    save_path=os.path.join(save_dir, "inter-response-timing.png"),
)

## 7. Stimulus Frequency Distributions

In [ ]:
plot_stimulus_frequency(
    primary_exps,
    title=f"{TASK_HR_NAME}: Stimulus Repeat Frequencies by ISI (N={analyses[primary_label]['N']})",
    save_path=os.path.join(save_dir, "stimulus-frequency-by-isi.png"),
)

## 8. Statistical Tests

In [ ]:
from scipy.stats import ttest_rel

# Paired t-test: ISI=16 vs ISI=64 (within the primary condition)
xf = x_primary[x_primary["isi"] != -1].copy()

dp16 = xf[xf["isi"] == 16].set_index("subject")["d_prime"]
dp64 = xf[xf["isi"] == 64].set_index("subject")["d_prime"]
paired = pd.concat([dp16, dp64], axis=1, keys=["dp16", "dp64"]).dropna()

if len(paired) > 0:
    import pandas as pd
    tval, pval = ttest_rel(paired["dp16"], paired["dp64"])
    print(f"Paired t-test ISI=16 vs ISI=64: t={tval:.3f}, {p_to_stars(pval)}")
    print(f"N paired = {paired.shape[0]}")
else:
    print("Not enough paired observations for ISI=16 vs ISI=64 test.")